# POC End-to-End Validation

Full pipeline run against one video clip with ground-truth comparison.

**Purpose:** Final accuracy check before field deployment.  
**Requires:** A labelled clip in `data/raw_footage/` and matching ground truth in `data/ground_truth/`.

Ground truth format (`data/ground_truth/{test_type}.json`):
```json
{
  "test_type": "explosiveness",
  "clips": {
    "sample_clip.mp4": {
      "students": [
        {"bib": 7,  "metric_value": 34.2, "metric_unit": "cm"},
        {"bib": 14, "metric_value": 28.8, "metric_unit": "cm"}
      ]
    }
  }
}
```

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

VIDEO_PATH = Path('../data/raw_footage/sample_clip.mp4')
TEST_TYPE  = 'explosiveness'
GT_PATH    = Path(f'../data/ground_truth/{TEST_TYPE}.json')
TARGET_FPS = 15
JOB_ID     = f'poc-validation-{TEST_TYPE}'

print(f"Video:        {VIDEO_PATH}  exists={VIDEO_PATH.exists()}")
print(f"Ground truth: {GT_PATH}  exists={GT_PATH.exists()}")

In [ ]:
# ── Load ground truth ────────────────────────────────────────────────────────
ground_truth = {}   # {bib: metric_value}
if GT_PATH.exists():
    with open(GT_PATH) as f:
        gt_data = json.load(f)
    clip_name = VIDEO_PATH.name
    students  = gt_data.get('clips', {}).get(clip_name, {}).get('students', [])
    ground_truth = {s['bib']: s['metric_value'] for s in students}
    print(f"Ground truth for {len(ground_truth)} students:")
    for bib, val in sorted(ground_truth.items()):
        print(f"  Bib {bib:3d}: {val} {students[0].get('metric_unit', '')}")
else:
    print("No ground truth file — will report results without accuracy metrics.")

In [ ]:
# ── Run full pipeline ─────────────────────────────────────────────────────────
import json
from worker.celery_app import _get_extractor
from pipeline.cache import PipelineCache
from pipeline.ingest import extract_frames
from pipeline.detect import PersonDetector
from pipeline.track import PersonTracker
from pipeline.pose import PoseEstimator
from pipeline.ocr import BibOCR, resolve_bibs
from pipeline.calibrate import Calibrator

cache = PipelineCache(job_id=JOB_ID, cache_root=Path('../data/cache'))

# Ingest
frames, frame_indices, timestamps = [], [], []
for fi, frame, ts in extract_frames(str(VIDEO_PATH), target_fps=TARGET_FPS):
    frames.append(frame); frame_indices.append(fi); timestamps.append(ts)

print(f"[1/6] Ingested {len(frames)} frames")

# Detect
detector = PersonDetector()
all_detections = []
for i, frame in enumerate(frames):
    dets = detector.detect(frame)
    for d in dets: d.frame_idx = frame_indices[i]
    all_detections.append(dets)
print(f"[2/6] Detected {sum(len(d) for d in all_detections)} persons")

# Track
tracker = PersonTracker()
all_tracks = []
for i, (frame, dets) in enumerate(zip(frames, all_detections)):
    all_tracks.append(tracker.update(dets, frame_idx=frame_indices[i]))
print(f"[3/6] Tracked — {len(set(t.track_id for ft in all_tracks for t in ft))} IDs")

# Pose
estimator = PoseEstimator()
all_poses = []
for frame, tracks in zip(frames, all_tracks):
    all_poses.append(estimator.estimate_batch(frame, tracks))
print(f"[4/6] Poses — {sum(len(p) for p in all_poses)} estimates")

# OCR
ocr = BibOCR()
frame_readings = []
for i, (frame, tracks) in enumerate(zip(frames, all_tracks)):
    if i % 5 == 0: frame_readings.append(ocr.read_frame(frame, tracks))
resolved = resolve_bibs(frame_readings)
for ft in all_tracks:
    for t in ft:
        t.bib_number, t.bib_confidence = resolved.get(t.track_id, (None, 0.0))
print(f"[5/6] OCR — resolved {len(resolved)} bibs: {resolved}")

# Calibrate
calibrator  = Calibrator()
calibration = calibrator.calibrate_single_axis(frames[0])
print(f"[6/6] Calibration — {calibration.method}  px/cm={calibration.pixels_per_cm}")

In [ ]:
# ── Extract metrics ───────────────────────────────────────────────────────────
configs_dir  = Path('../configs/test_configs')
config_file  = configs_dir / f"{TEST_TYPE}.json"
geometry_cfg = json.loads(config_file.read_text()) if config_file.exists() else {}

extractor = _get_extractor(TEST_TYPE, geometry_cfg, calibration)
results   = extractor.extract(all_tracks, all_poses, frames)
cache.save_results(results)

print(f"\n=== {TEST_TYPE.upper()} RESULTS ===")
for r in results:
    bib = r.student_bib if r.student_bib and r.student_bib != -1 else f"(track {r.track_id})"
    print(f"  Bib {str(bib):>4s}  {r.metric_value:.2f} {r.metric_unit}  "
          f"conf={r.confidence_score:.2f}  flags={r.flags or '—'}")

In [ ]:
# ── Accuracy analysis ─────────────────────────────────────────────────────────
if ground_truth:
    errors = []
    rows   = []
    for r in results:
        gt = ground_truth.get(r.student_bib)
        if gt is not None:
            err = r.metric_value - gt
            errors.append(abs(err))
            rows.append((r.student_bib, r.metric_value, gt, err, r.confidence_score))

    if rows:
        print(f"\n{'Bib':>5} {'Pred':>8} {'GT':>8} {'Error':>8} {'Conf':>6}")
        print("─" * 42)
        for bib, pred, gt, err, conf in sorted(rows):
            flag = "✓" if abs(err) <= 2.0 else "✗"
            print(f"{bib:>5} {pred:>8.2f} {gt:>8.2f} {err:>+8.2f} {conf:>6.2f}  {flag}")
        print(f"\nMAE: {np.mean(errors):.3f}  Max: {max(errors):.3f}  n={len(errors)}")

        fig, ax = plt.subplots(figsize=(8, 4))
        bibs_r  = [r[0] for r in rows]
        pred_v  = [r[1] for r in rows]
        gt_v    = [r[2] for r in rows]
        x = np.arange(len(rows))
        ax.bar(x - 0.2, pred_v, 0.4, label='Predicted', color='steelblue')
        ax.bar(x + 0.2, gt_v,   0.4, label='Ground Truth', color='tomato')
        ax.set_xticks(x); ax.set_xticklabels([f"Bib {b}" for b in bibs_r])
        ax.set_ylabel(results[0].metric_unit if results else '')
        ax.set_title(f'{TEST_TYPE} — Predicted vs Ground Truth')
        ax.legend(); plt.tight_layout(); plt.show()
    else:
        print("No overlap between results bibs and ground truth bibs.")
else:
    print("No ground truth available — skipping accuracy analysis.")

In [ ]:
# ── Save annotated validation video ──────────────────────────────────────────
from pipeline.visualise import PipelineVisualiser, VisOptions

output_path = Path(f'../data/annotated/poc_validation_{TEST_TYPE}.mp4')
output_path.parent.mkdir(parents=True, exist_ok=True)

opts = VisOptions(show_calibration_grid=calibration.is_valid)
with PipelineVisualiser(output_path, test_type=TEST_TYPE, fps=TARGET_FPS, options=opts) as vis:
    for frame, tracks, poses, ts in zip(frames, all_tracks, all_poses, timestamps):
        vis.write_frame(frame, tracks, poses, calibration, results, timestamp_s=ts)

print(f"Annotated validation video saved → {output_path}")